In [ ]:
from stock_predictor.utils import (
    create_s3_client,
    get_latest_data_s3,
)

client = create_s3_client()

## Loading data from MinIO

We retrieve the latest candles for a given ticker from MinIO object storage
and inspect the first rows to verify the schema and column types are correct.

In [ ]:
df = get_latest_data_s3(client, "TSLA")
print(df.head())

**Finding:** Schema and column types look correct. Sentiment columns are null 
for the earliest dates, as no news were available for those timestamps.
This is expected behavior and will be handled during feature engineering.

## Descriptive statistics

We compute summary statistics for all numeric columns to detect anomalies
such as negative prices, zero volume days, or sentiment scores out of range.

In [ ]:
df.describe()

**Finding:** Price columns (open, high, low, close) show no negative values.
Volume has a high standard deviation, indicating significant variation across
sessions. Sentiment compound scores stay within the expected [-1, 1] range.
No critical anomalies detected.

## Missing values

We check for null values across all columns to understand data completeness
before building features. Null sentiment scores are expected for early dates
where no news were available.

In [ ]:
df.isnull().sum()

**Finding:** Price and volume columns are complete with no nulls. 
Sentiment columns (neg, neu, pos, compound) show nulls only for the 
earliest timestamps, consistent with the news coverage gap identified above.
These will be forward-filled or zero-filled in features.py.

## Temporal consistency

We check how many 10-minute candles exist per day. A normal US market
session has ~39 candles (09:30–16:00). Days with fewer candles indicate
early closes or collector failures.

In [ ]:
df.resample("1D").size().plot()

**Finding:** Weekdays consistently show ~95 candles. The small dips around
January 2025 correspond to NYSE holidays, not pipeline failures. Weekends
correctly appear as gaps in the index.

**Impact on features:** When building lags, the first Monday candle will
correctly point to the last Friday candle with no leakage risk.

## Identifying incomplete trading days

We filter out weekends and flag weekdays with fewer than 80 candles.
These are candidates for early market closes (e.g. day before Thanksgiving)
or collector failures that could introduce gaps in the lag features.

In [ ]:
daily = df.resample("1D").size()
daily = daily[daily.index.dayofweek < 5]
daily[daily < 80].sort_values()

**Finding:** All flagged dates correspond to known NYSE holidays and early closes.
No collector failures detected on regular trading days.

- **Zero-candle days (0):** Federal holidays — Independence Day, Labor Day,
  MLK Day, Presidents Day, Good Friday, Juneteenth, Christmas, New Year's Day.
- **Half-sessions with ~6 candles:** Thanksgiving Day and New Year's Day 2025
  (market opened briefly).
- **Early closes with ~61-67 candles:** Day before Thanksgiving, Christmas Eve,
  and July 3rd — NYSE closes at 13:00 on these dates.

These dates will be flagged with a `is_short_session` boolean feature in
features.py, as price behavior during half-sessions differs from full days.

## Sentiment score distribution

We plot the distribution of the compound sentiment score (range [-1, 1])
to understand how news sentiment is spread across the dataset.
A score near -1 indicates strongly negative news, +1 strongly positive,
and 0 neutral or no news available.

In [ ]:
df["sentiment"].hist(bins=50)

**Finding:** The distribution is strongly bimodal with large spikes at -1 and +1,
and a smaller peak at 0. This suggests that most news articles carry a clear
strong sentiment rather than a neutral tone. The 0 spike corresponds to
timestamps with no news available (null-filled as 0).

**Impact on features:** Given the bimodal nature, the raw compound score may
already act as a near-binary signal. Consider adding a `sentiment_extreme`
boolean feature (abs(compound) > 0.8) in features.py to explicitly capture
this pattern, in addition to the raw score.

## News coverage frequency

We count the number of unique headlines to understand how often news
updates relative to the number of candles in the dataset.

In [ ]:
df["headline"].nunique()

**Finding:** Only 2,639 unique headlines across the full dataset, meaning
each news article covers multiple candles on average. The same headline
stays active for several 10-minute intervals until a new article is fetched.

**Impact on features:** The raw sentiment score will be highly autocorrelated
since it stays constant across consecutive candles. A rolling mean of sentiment
over the last 3-5 candles adds little new information in this case. Instead,
features.py should flag the **moment a headline changes** (`news_changed` boolean)
as that transition point is likely more informative than the score itself.

## Price return autocorrelation

We compute the lag-1 autocorrelation of 10-minute returns to check whether
past returns are predictive of future returns. A value near 0 indicates
no linear dependency, while a high value would suggest strong momentum.

In [ ]:
df["c"].pct_change().autocorr(lag=1)

**Finding:** Lag-1 autocorrelation is virtually zero (0.006), confirming that
raw returns have no meaningful linear dependency at the 10-minute level.
This is consistent with the efficient market hypothesis for liquid assets.

**Impact on features:** Lagged raw returns alone will not be useful features.
The model will need to rely on non-linear combinations of price features
(ATR, RSI, VWAP deviation) and sentiment signals to find predictive patterns.
This also reinforces why we start with a Ridge baseline — if it performs
poorly, it confirms the signal is non-linear and justifies moving to LightGBM.